# Capítulo 7: Explicabilidad de Modelos de IA (XAI) *(Versión Simulada)*

## 7.1. Importancia en ciberseguridad

Este notebook re-entrena el modelo de detección de malware sobre datos simulados y aplica **SHAP** para explicar las predicciones. Esto permite validar que las características más importantes coincidan con lo esperado (alta entropía, pocas importaciones → malware).

- **Auditorías de cumplimiento** (GDPR, NIS2).
- **Generación de evidencia** digital forense.
- **Reducción de falsos positivos** validados por analistas.

---
## 7.2. SHAP: SHapley Additive exPlanations

### 7.2.1. Carga de datos y entrenamiento del modelo

In [ ]:
# Listing 7.1: Explicación de predicciones con SHAP

import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# 1. Entrenar modelo con datos simulados de malware
df = pd.read_csv('data/file_features.csv').dropna()
X, y = df.drop('label', axis=1), df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

print(f"[OK] Modelo Random Forest entrenado sobre datos simulados.")
print(f"     Accuracy: {rf.score(X_test, y_test):.4f}")
print(f"     Características: {list(X.columns)}")

In [ ]:
# 2. Calcular valores SHAP
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

print(f"[OK] Valores SHAP calculados para {len(X_test)} muestras de prueba.")
print(f"\n     Las características con mayor valor SHAP contribuyen más")
print(f"     a clasificar un archivo como MALICIOSO.")

In [ ]:
# 3. Gráfico de resumen (importancia global - barras)
shap.summary_plot(
    shap_values[1], X_test,
    plot_type='bar',
    show=False
)
plt.title('Importancia SHAP - Detección de Malware (Simulado)')
plt.tight_layout()
plt.savefig('data/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Gráfico de puntos (distribución de impacto)
shap.summary_plot(
    shap_values[1], X_test,
    show=False
)
plt.title('SHAP - Distribución de impacto por característica (Simulado)')
plt.tight_layout()
plt.savefig('data/shap_dot_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4. Explicar una muestra individual (maliciosa)
muestra_idx = 0

# Buscar primera muestra clasificada como maliciosa
preds_test = rf.predict(X_test)
idx_mal = np.where(preds_test == 1)[0]
if len(idx_mal) > 0:
    muestra_idx = idx_mal[0]

shap.force_plot(
    explainer.expected_value[1],
    shap_values[1][muestra_idx, :],
    X_test.iloc[muestra_idx, :],
    matplotlib=True,
    show=False
)
plt.title(f'Explicación individual - Muestra {muestra_idx} (Maliciosa)')
plt.tight_layout()
plt.savefig('data/shap_force_plot.png', dpi=150, bbox_inches='tight')
plt.show()

pred = rf.predict(X_test.iloc[[muestra_idx]])[0]
proba = rf.predict_proba(X_test.iloc[[muestra_idx]])[0]
print(f"\nMuestra {muestra_idx}:")
print(f"  Predicción: {'Malicious' if pred == 1 else 'Benign'}")
print(f"  Probabilidad → Benign: {proba[0]:.3f} | Malicious: {proba[1]:.3f}")
print(f"\n  Valores de características:")
for col, val in X_test.iloc[muestra_idx].items():
    print(f"    {col:<25}: {val}")

### 7.2.2. Análisis de dependencia SHAP

Muestra cómo el valor de la característica más importante afecta la predicción del modelo.

In [ ]:
# Dependencia para las 2 características más importantes
mean_shap = np.abs(shap_values[1]).mean(axis=0)
top_features = X_test.columns[np.argsort(mean_shap)[::-1][:2]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, feat in enumerate(top_features):
    plt.sca(axes[i])
    shap.dependence_plot(feat, shap_values[1], X_test, ax=axes[i], show=False)
    axes[i].set_title(f'Dependencia SHAP: {feat}')

plt.tight_layout()
plt.savefig('data/shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\n[INFO] Las 2 características más importantes son: {list(top_features)}")

---
## 7.3. Resumen de artefactos

In [ ]:
print("=== Artefactos XAI generados ===")
print("  data/shap_summary.png       - Importancia global (barras)")
print("  data/shap_dot_summary.png   - Distribución de impacto")
print("  data/shap_force_plot.png    - Explicación individual")
print("  data/shap_dependence.png    - Gráficos de dependencia")
print("\n[OK] Capítulo 7 completado sobre datos simulados.")